# AwareLiquid · Phase 5b — Qwen-2.5-1.5B + MT adapter (Kaggle T4)

Re-run of Phase 5 on a stronger base (Qwen-2.5-1.5B-Instruct) so that needle-in-haystack has non-zero base scores to compare against.

**Settings vs Phase 5:**
- MODEL: `Qwen/Qwen2.5-1.5B-Instruct` (1.5 B params, ~3 GB in fp16)
- SEQ_LEN: 512 (tighter than TinyLlama's 768 — Qwen is 40 % bigger)
- everything else identical (1000 steps, batch 1, grad_accum 8, MT every 4th layer, LoRA on q/k/v/o)

**Hardware:** GPU T4 (14.6 GB), Internet on. Wall-clock: ~3-4 h.

The smoke test in `scripts/cloud_llama_mt_experiment.sh` triggers a known stub-only false-positive AssertionError — we sed-patch it out before invoking the script (same approach as Phase 5).

## 1 · GPU sanity

In [ ]:
import torch, platform
print('python', platform.python_version())
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('mem_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
assert torch.cuda.is_available(), 'GPU required'

## 2 · Clone repo

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/everest-an/M1.git'
REPO_DIR = '/kaggle/working/M1'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
subprocess.check_call(['git', 'log', '-1', '--oneline'])

## 3 · Install dependencies

In [ ]:
!pip install -q -r requirements.txt accelerate safetensors peft datasets

## 4 · Adapter import check (skips brittle pytest)

In [ ]:
!python -c "from mt_lnn.llama_adapter import attach_mt_adapters, count_trainable_parameters; print('adapter import OK')"

## 5 · Train + PPL + Needle (Qwen-2.5-1.5B)

In [ ]:
%env MODEL=Qwen/Qwen2.5-1.5B-Instruct
%env SEQ_LEN=512
%env BATCH=1
%env GRAD_ACCUM=8
%env STEPS=1000
%env MT_EVERY=4
%env NEEDLE_CONTEXTS=1024 2048 4096
%env NEEDLE_SAMPLES=5
%env OUT_DIR=/kaggle/working/checkpoints/qwen_mt_adapter
%env RESULT_DIR=/kaggle/working/benchmarks/kaggle_qwen_run
!sed -i 's|python -m pytest tests/test_llama_adapter.py -q|echo "skipping smoke test (known stub-only false positive)"|' scripts/cloud_llama_mt_experiment.sh
!bash scripts/cloud_llama_mt_experiment.sh

## 6 · Package artefacts

In [ ]:
import shutil, glob
from pathlib import Path
out_root = Path('/kaggle/working/awareliquid_phase5b_artifacts')
out_root.mkdir(parents=True, exist_ok=True)
for ckpt in sorted(glob.glob('/kaggle/working/checkpoints/qwen_mt_adapter/*.pt'))[-2:]:
    shutil.copy(ckpt, out_root / Path(ckpt).name)
for j in glob.glob('/kaggle/working/benchmarks/kaggle_qwen_run/*.json'):
    shutil.copy(j, out_root / Path(j).name)
for j in glob.glob('/kaggle/working/benchmarks/kaggle_qwen_run/*.log'):
    shutil.copy(j, out_root / Path(j).name)
archive = shutil.make_archive('/kaggle/working/awareliquid_phase5b', 'zip', out_root)
print('archive:', archive)
print('size MB:', round(Path(archive).stat().st_size / 1024**2, 1))

## 7 · Eyeball results

In [ ]:
import json, pathlib
for name in ('ppl_ablation.json', 'needle.json'):
    p = pathlib.Path('/kaggle/working/benchmarks/kaggle_qwen_run') / name
    if p.exists():
        print(f'--- {name} ---')
        print(json.dumps(json.loads(p.read_text()), indent=2)[:2000])